# 2. LangChain Model API - Abstracción y Framework

## Objetivos de Aprendizaje
- Comprender las ventajas del framework LangChain sobre APIs directas
- Configurar `ChatGroq` con diferentes parámetros
- Explorar la compatibilidad entre diferentes modelos
- Implementar patrones de uso común con LangChain

## Introducción a LangChain

LangChain es un framework que simplifica el desarrollo de aplicaciones con modelos de lenguaje. Principales ventajas:
- **Abstracción**: Una interfaz unificada para múltiples proveedores
- **Herramientas**: Componentes predefinidos para tareas comunes
- **Cadenas**: Composición de múltiples operaciones
- **Memoria**: Gestión automática del historial de conversaciones

## Instalación de Dependencias
```bash
pip install langchain langchain-groq
```

> Necesitas tu `GROQ_API_KEY` configurada (ver notebook `1-groq_model_api.ipynb`, "Paso 0").

In [1]:
# Importar las bibliotecas de LangChain
import os

# Carga de credenciales: funciona igual en Google Colab y en local (.env)
try:
    from google.colab import userdata  # type: ignore
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()

assert os.getenv("GROQ_API_KEY"), "Falta GROQ_API_KEY (Colab: Secrets · local: archivo .env)"

MODELO = os.getenv("GROQ_MODEL", "openai/gpt-oss-120b")

from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

# Verificar versiones
print("Verificando instalación de LangChain...")
try:
    import langchain
    print(f"✓ LangChain version: {langchain.__version__}")
except ImportError:
    print("✗ LangChain no está instalado")

print("Bibliotecas importadas correctamente")

Verificando instalación de LangChain...
✓ LangChain version: 1.2.14
Bibliotecas importadas correctamente


In [2]:
# Configuración del modelo LangChain con Groq
# ChatGroq lee GROQ_API_KEY del entorno automáticamente
try:
    llm = ChatGroq(
        model=MODELO,
        temperature=0.7,
        max_tokens=150,
        reasoning_effort="low"
    )

    print("✓ Modelo LangChain configurado correctamente")
    print(f"Modelo: {llm.model_name}")
    print(f"Temperature: {llm.temperature}")
    print(f"Max tokens: {llm.max_tokens}")

except Exception as e:
    print(f"✗ Error en configuración: {e}")
    print("Verifica la variable de entorno GROQ_API_KEY")

✓ Modelo LangChain configurado correctamente
Modelo: openai/gpt-oss-120b
Temperature: 0.7
Max tokens: 150


In [3]:
# Uso básico con LangChain - Diferentes tipos de mensajes
def ejemplo_basico():
    try:
        # Usar HumanMessage (equivalente a "user" en OpenAI)
        response = llm.invoke([HumanMessage(content="Hola, ¿cómo estás?")])
        print("=== Respuesta Básica ===")
        print(response.content)
        print(f"Tipo de respuesta: {type(response)}")
        
    except Exception as e:
        print(f"Error: {e}")

# Ejecutar ejemplo básico
ejemplo_basico()

=== Respuesta Básica ===
**Hola, estoy bien, gracias.** Es un placer conocerte. ¿En qué puedo ayudarte hoy? ¿Tienes alguna pregunta o necesitas ayuda con algo en particular? Estoy aquí para ayudarte en lo que necesites.
Tipo de respuesta: <class 'langchain_core.messages.ai.AIMessage'>


## Configuración Avanzada con LangChain

LangChain permite configuraciones más sofisticadas y cambiar proveedores fácilmente.

In [4]:
# Configuraciones múltiples con diferentes parámetros
def configuraciones_multiples():
    # Configuración conservadora (para tareas que requieren precisión)
    llm_conservador = ChatGroq(
        model=MODELO,
        temperature=0.1,  # Muy determinístico
        max_tokens=100,
        reasoning_effort="low"
    )

    # Configuración creativa (para tareas que requieren creatividad)
    llm_creativo = ChatGroq(
        model=MODELO,
        temperature=0.9,  # Muy creativo
        max_tokens=150,
        reasoning_effort="low"
    )

    prompt = "Escribe un eslogan para una empresa de tecnología"

    print("=== COMPARACIÓN DE CONFIGURACIONES ===")

    try:
        # Respuesta conservadora
        print("\n1. Configuración Conservadora (temp=0.1):")
        print("-" * 40)
        response_conservador = llm_conservador.invoke([HumanMessage(content=prompt)])
        print(response_conservador.content)

        # Respuesta creativa
        print("\n2. Configuración Creativa (temp=0.9):")
        print("-" * 35)
        response_creativo = llm_creativo.invoke([HumanMessage(content=prompt)])
        print(response_creativo.content)

    except Exception as e:
        print(f"Error: {e}")

# Ejecutar comparación
configuraciones_multiples()

=== COMPARACIÓN DE CONFIGURACIONES ===

1. Configuración Conservadora (temp=0.1):
----------------------------------------
"Conecta con el futuro, innova con nosotros"

2. Configuración Creativa (temp=0.9):
-----------------------------------
"Conecta tu futuro, innova tu presente"


## Comparación: LangChain vs SDK de Groq directo

Veamos las diferencias en código entre usar LangChain y el cliente `groq` directo:

In [5]:
# Comparación de código entre LangChain y el SDK de Groq directo
from groq import Groq

def comparar_enfoques():
    prompt = "Explica qué es Python en una oración"

    print("=" * 60)
    print("COMPARACIÓN: LangChain vs SDK de Groq directo")
    print("=" * 60)

    # Método 1: SDK de Groq directo
    print("\n1. SDK de Groq directo:")
    print("-" * 20)
    try:
        cliente = Groq()  # lee GROQ_API_KEY del entorno

        respuesta_sdk = cliente.chat.completions.create(
            model=MODELO,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.7,
            max_tokens=50
        )

        print(f"Respuesta: {respuesta_sdk.choices[0].message.content}")
        print(f"Tokens: {respuesta_sdk.usage.total_tokens}")

    except Exception as e:
        print(f"Error SDK Groq: {e}")

    # Método 2: LangChain
    print("\n2. LangChain:")
    print("-" * 15)
    try:
        response_langchain = llm.invoke([HumanMessage(content=prompt)])
        print(f"Respuesta: {response_langchain.content}")
        print(f"Tipo: {type(response_langchain)}")

    except Exception as e:
        print(f"Error LangChain: {e}")

    print("\n" + "=" * 60)
    print("VENTAJAS DE CADA ENFOQUE:")
    print("=" * 60)
    print("SDK de Groq directo:")
    print("+ Control total sobre parámetros")
    print("+ Acceso directo a metadatos (tokens, tiempos)")
    print("+ Menor abstracción, más transparente")
    print()
    print("LangChain:")
    print("+ Interfaz unificada para múltiples proveedores")
    print("+ Herramientas adicionales (cadenas, memoria, etc.)")
    print("+ Más fácil composición de operaciones complejas")
    print("+ Mejor para prototipado rápido")

# Ejecutar comparación
comparar_enfoques()

COMPARACIÓN: LangChain vs SDK de Groq directo

1. SDK de Groq directo:
--------------------
Respuesta: Python es un lenguaje de programación de alto nivel, interpretado y orientado a objetos, ampliamente utilizado para desarrollar aplicaciones web, analizar datos, crear inteligencia artificial y realizar tareas de automatización, gracias a
Tokens: 94

2. LangChain:
---------------
Respuesta: Python es un lenguaje de programación de alto nivel, interpretado y orientado a objetos, ampliamente utilizado para desarrollar aplicaciones web, análisis de datos, inteligencia artificial y más, gracias a su sintaxis simple y fácil de aprender.
Tipo: <class 'langchain_core.messages.ai.AIMessage'>

VENTAJAS DE CADA ENFOQUE:
SDK de Groq directo:
+ Control total sobre parámetros
+ Acceso directo a metadatos (tokens, tiempos)
+ Menor abstracción, más transparente

LangChain:
+ Interfaz unificada para múltiples proveedores
+ Herramientas adicionales (cadenas, memoria, etc.)
+ Más fácil composición de o

## Tipos de Mensajes en LangChain

LangChain proporciona diferentes tipos de mensajes que corresponden a los roles de la API de chat:
- **HumanMessage**: Mensajes del usuario (equivale a `"user"`)
- **AIMessage**: Respuestas del asistente (equivale a `"assistant"`)
- **SystemMessage**: Instrucciones del sistema (equivale a `"system"`)

In [6]:
# Ejemplo con múltiples tipos de mensajes
def ejemplo_conversacion_completa():
    try:
        messages = [
            SystemMessage(content="Eres un tutor de programación amigable y paciente. Explicas conceptos técnicos de forma clara y das ejemplos prácticos."),
            HumanMessage(content="¿Qué es una función en programación?"),
            AIMessage(content="Una función es un bloque de código reutilizable que realiza una tarea específica. Te ayuda a organizar tu código y evitar repetición."),
            HumanMessage(content="¿Puedes darme un ejemplo simple en Python?")
        ]
        
        response = llm.invoke(messages)
        print("=== Conversación con Contexto ===")
        print(response.content)
        
    except Exception as e:
        print(f"Error: {e}")

# Ejecutar ejemplo
ejemplo_conversacion_completa()

=== Conversación con Contexto ===
Claro, aquí tienes un ejemplo de una función simple en Python:

```python
def saluda(nombre):
  print("Hola, " + nombre)

saluda("Juan")
```

En este ejemplo, `saluda` es una función que toma un parámetro `nombre` y lo usa para imprimir un saludo personalizado. Cuando llamamos a `saluda("Juan")`, la función imprime "Hola, Juan". Esto demuestra cómo puedes reutilizar el código de la función pasándole diferentes parámetros.


## Ejercicios Prácticos

### Ejercicio 1: Crear Diferentes Personalidades
Usa SystemMessage para crear asistentes con diferentes personalidades (formal, casual, técnico, creativo).

### Ejercicio 2: Cadena de Conversación
Construye una conversación de múltiples turnos usando los diferentes tipos de mensajes.

### Ejercicio 3: Comparar Modelos
Configura un segundo `ChatGroq` con `openai/gpt-oss-20b` y compara calidad y velocidad frente a `openai/gpt-oss-120b`.

## Conceptos Clave Aprendidos

1. **Abstracción de LangChain** sobre APIs directas
2. **Tipos de mensajes** y su equivalencia con los roles de la API de chat
3. **Configuraciones múltiples** para diferentes casos de uso
4. **Ventajas y desventajas** de frameworks vs APIs directas
5. **Intercambiabilidad** de proveedores con LangChain

## Próximos Pasos

En el siguiente notebook exploraremos el **streaming** con LangChain, que permite mostrar respuestas en tiempo real para mejorar la experiencia de usuario.